# **Dados e Aprendizagem Automática** 

## **NaiveBayes - Tratamento com Grid Search**

Nesta abordagem, utilizámos o pré-processamento definido anteriormente (Tratamento 2), incluindo a extração de características temporais e a limpeza das variáveis categóricas. Para as variáveis categóricas, aplicámos **OneHotEncoder** de forma a evitar relações ordinais artificiais que poderiam prejudicar a aprendizagem da rede neural.  

Na fase de modelação, construímos um **pipeline** que combina o pré-processamento com o **MLPClassifier**. Para otimizar o desempenho, aplicámos **Grid Search** sobre os principais hiperparâmetros do modelo, incluindo o número de neurónios nas camadas ocultas, a função de ativação, a taxa de aprendizagem inicial e o parâmetro de regularização (`alpha`). Esta abordagem permite encontrar a configuração que melhor se adapta aos dados, garantindo uma aprendizagem robusta e generalizável.


**Imports necessários para este teste**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

%matplotlib inline

### **Preparation**
**Load the CSVs**

In [ ]:
df_train = pd.read_csv('../../Datasets/training_data.csv', encoding='latin-1')
df_test = pd.read_csv('../../Datasets/test_data.csv', encoding='latin-1')

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

**Feature Engineering (Extração da data)**

In [3]:
def extract_date_features(df):
    df['record_date'] = pd.to_datetime(df['record_date'])
    df['hour'] = df['record_date'].dt.hour
    df['day_of_week'] = df['record_date'].dt.dayofweek # Monday=0, Sunday=6
    df['month'] = df['record_date'].dt.month
    
    # Create "Weekend" feature
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    
    # Create "Rush Hour" feature (7 da manhã até às 9 da manhã e 4 da tarde ate às 7 da tarde, podemos brincar com estas horas)
    df['is_rush_hour'] = df['hour'].apply(lambda x: 1 if (7 <= x <= 9) or (16 <= x <= 19) else 0)
    
    return df.drop(columns=['record_date'])

df_train = extract_date_features(df_train)
df_test = extract_date_features(df_test)

**Missing Values e Valores Incorretos**

In [4]:
def clean_categorical_text(df):

    # Primeiro "limpamos" a coluna 'AVERAGE CLOUDINESS'
    correcoes_erros = {
        'cï¿½u': 'ceu',      # erro especifico
        'c\u00e9u': 'ceu', # é
        'c\u00fa': 'ceu',  # ú
        'c\u00fau': 'ceu', 
        'céu': 'ceu'
    }
    # regex=True permite substituir apenas parte da frase (ex: "cï¿½u claro" -> "ceu claro")
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].astype(str).replace(correcoes_erros, regex=True)

    cloud_map = {
        'ceu claro': 'ceu_claro',
        'ceu limpo': 'ceu_claro',

        'ceu pouco nublado': 'pouco_nublado',
        'nuvens dispersas': 'pouco_nublado',
        'algumas nuvens': 'pouco_nublado',

        'nuvens quebrados': 'nublado', 
        'nuvens quebradas': 'nublado',
        'tempo nublado': 'nublado',
        'nublado': 'nublado',
    }
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].astype(str).replace(cloud_map, regex=True)
    # Tratamos também dos seus missing values
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].replace('nan', 'unknown_cloudiness')
    
    # Depois "limpamos" também a coluna da "AVERAGE RAIN"
    rain_map = {
        'chuva fraca': 'chuva_fraca',
        'chuva leve': 'chuva_fraca',
        'aguaceiros fracos': 'chuva_fraca',
        'chuvisco fraco': 'chuva_fraca',
        'chuvisco e chuva fraca': 'chuva_fraca',
        'trovoada com chuva leve': 'chuva_fraca', 

        'chuva moderada': 'chuva_moderada',
        'chuva': 'chuva_moderada',
        'aguaceiros': 'chuva_moderada',
        'trovoada com chuva': 'chuva_moderada',

        'chuva forte': 'chuva_forte',
        'chuva de intensidade pesada': 'chuva_forte',
        'chuva de intensidade pesado': 'chuva_forte'
    }
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].replace(rain_map)
    # Tratamos também dos seus missing values
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].fillna('no_rain')
    
    #df["LUMINOSITY"] = df_train["LUMINOSITY"].replace("LOW_LIGHT", "LIGHT")
    
    return df

df_train = clean_categorical_text(df_train)
df_test = clean_categorical_text(df_test)

In [5]:
df_train["AVERAGE_SPEED_DIFF"] = df_train["AVERAGE_SPEED_DIFF"].fillna("None")

**Verificação dos valores dessas colunas agora**

In [ ]:
df_test['AVERAGE_CLOUDINESS'].value_counts()

In [ ]:
df_test['AVERAGE_RAIN'].value_counts()

**Drop de Colunas Redundates** 

Considera-mo-las redundantes devido a 'CITY_NAME' conter um valor constante ("Porto") e 'AVERAGE_PRECIPITATION' consistir quase apenas em zeros.

In [8]:
cols_to_drop = ['city_name', 'AVERAGE_PRECIPITATION']
df_train = df_train.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

**Handling categoric data**

In [ ]:
# Identify categorical and numerical features

categorical_features = [
    "AVERAGE_RAIN",
    "AVERAGE_CLOUDINESS",
    "LUMINOSITY"
]

numerical_features = [
    col for col in df_train.columns
    if col not in categorical_features + ["AVERAGE_SPEED_DIFF"]
]

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)


# ColumnTransformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

In [ ]:
df_train.info()

In [ ]:
df_test.info()

### **Modeling**

Select features and target

In [12]:
X_train = df_train.drop(columns=["AVERAGE_SPEED_DIFF"])
y_train = df_train["AVERAGE_SPEED_DIFF"]

In [13]:
X_test = df_test

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

Train/Test Split

In [14]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=2022, stratify=y_train
)

Definir a Pipeline

In [ ]:
nb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("nb", GaussianNB())
])

Parâmetros da Grid Search

In [ ]:
param_grid = {
    "nb__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]
}

GridSearchCV

In [ ]:
grid = GridSearchCV(
    estimator=nb_pipeline,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2,
    refit=True
)

Fit

In [ ]:
grid.fit(X_tr, y_tr)

Avaliar no teste

In [ ]:
val_preds = grid.predict(X_val)

print("Best Parameters:", grid.best_params_)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("\nClassification Report:\n", classification_report(y_val, val_preds))

Treinar o modelo

In [ ]:
best_nb = grid.best_estimator_
best_nb.fit(X_train, y_train)

Predicts

In [ ]:
preds = best_nb.predict(X_test)

Para submeter no kaggle

In [ ]:
submission = pd.DataFrame({
    "RowId": range(1, len(preds)+1),
    "Speed_Diff": preds
})

submission.to_csv("../../results/NaiveBayes/nb2grid.csv", index=False)
submission.head()